# 03 — Assembler, CPU Emulator, Boot Image, and Relocations

**Modules:** `03.01`, `03.02`, `03.03`, `04.02`  
**Prerequisite:** notebooks 01–02; instruction words inherit the byte contract and `OUT` inherits the UART boundary.  
**Consumes:** `01-byte-contract.json`, `02-uart-trace.json`  
**Emits:** `fromthetransistor/03-machine-image.json`

This fixed-width toy ISA is deliberately inspectable: every instruction is two bytes, labels resolve to byte addresses, and a CPU step changes only declared state. It is not ARM and it is not an ELF implementation. It isolates the contracts that a real assembler, emulator, boot monitor, and linker must agree on.

In [ ]:
from pathlib import Path
import json
import os

ARTIFACT_ROOT = Path(os.environ['COURSE_NOTEBOOK_ARTIFACTS'])
byte_contract = json.loads((ARTIFACT_ROOT / 'fromthetransistor/01-byte-contract.json').read_text(encoding='utf-8'))
uart_contract = json.loads((ARTIFACT_ROOT / 'fromthetransistor/02-uart-trace.json').read_text(encoding='utf-8'))
assert byte_contract['byteorder'] == 'little'
assert uart_contract['format'] == '8N1-lsb-first'
assert uart_contract['payload_hex'] == byte_contract['known_word_hex']
OP = {'MOVI': 1, 'ADD': 2, 'SUBI': 3, 'JNZ': 4, 'OUT': 5, 'HALT': 15}
OP_NAME = {value: key for key, value in OP.items()}


## Worked analysis

The first assembler pass assigns labels; the second emits opcodes and resolves operands. The emulator fetches at a byte PC, decodes two bytes, executes, and records a monitor trace. A tiny absolute relocation shows why symbol value, patch offset, width, and byte order are all part of an object-format contract. The program sums integers 1 through 5 and sends `15` to the inherited UART-facing output.

In [ ]:
def register(token):
    if not token.startswith('r') or not token[1:].isdigit():
        raise ValueError(f'invalid register {token}')
    number = int(token[1:])
    if not 0 <= number < 4:
        raise ValueError('toy CPU has registers r0-r3')
    return number

def clean_lines(source):
    return [line.split('#', 1)[0].strip() for line in source.splitlines() if line.split('#', 1)[0].strip()]

def assemble(source):
    lines = clean_lines(source)
    labels = {}
    pc = 0
    for line in lines:
        if line.endswith(':'):
            name = line[:-1]
            if name in labels:
                raise ValueError(f'duplicate label {name}')
            labels[name] = pc
        else:
            pc += 2
    image = bytearray()
    for line in lines:
        if line.endswith(':'):
            continue
        parts = line.replace(',', ' ').split()
        name = parts[0].upper()
        if name not in OP:
            raise ValueError(f'unknown instruction {name}')
        a = 0
        operand = 0
        if name in {'MOVI', 'SUBI'}:
            a, operand = register(parts[1]), int(parts[2], 0)
        elif name == 'ADD':
            a, operand = register(parts[1]), register(parts[2])
        elif name == 'JNZ':
            a = register(parts[1])
            if parts[2] not in labels:
                raise ValueError(f'unresolved label {parts[2]}')
            operand = labels[parts[2]]
        elif name == 'OUT':
            a = register(parts[1])
        if not 0 <= operand <= 0xFF:
            raise OverflowError('operand does not fit one byte')
        image.extend([(OP[name] << 4) | a, operand])
    return bytes(image), labels

def run(image, max_steps=100):
    registers = [0, 0, 0, 0]
    pc = 0
    output = []
    trace = []
    for step in range(max_steps):
        if pc + 1 >= len(image):
            raise RuntimeError('fetch outside image')
        first, operand = image[pc], image[pc + 1]
        opcode, a = first >> 4, first & 0x0F
        name = OP_NAME.get(opcode)
        if name is None or a >= len(registers):
            raise RuntimeError('illegal instruction')
        trace.append({'step': step, 'pc': pc, 'op': name, 'registers': registers.copy()})
        next_pc = pc + 2
        if name == 'MOVI': registers[a] = operand
        elif name == 'ADD': registers[a] = (registers[a] + registers[operand]) & 0xFFFF
        elif name == 'SUBI': registers[a] = (registers[a] - operand) & 0xFFFF
        elif name == 'JNZ' and registers[a] != 0: next_pc = operand
        elif name == 'OUT': output.append(registers[a] & 0xFF)
        elif name == 'HALT': return {'registers': registers, 'output': output, 'trace': trace}
        pc = next_pc
    raise RuntimeError('step limit exceeded')

program = '''
MOVI r0, 5
MOVI r1, 0
loop:
ADD r1, r0
SUBI r0, 1
JNZ r0, loop
OUT r1
HALT
'''
image, labels = assemble(program)
result = run(image)
assert len(image) == 14
assert labels['loop'] == 4
assert result['registers'][1] == 15
assert result['output'] == [15]

def patch_abs16(buffer, offset, symbol_value):
    if not 0 <= symbol_value <= 0xFFFF or not 0 <= offset <= len(buffer) - 2:
        raise ValueError('invalid relocation')
    buffer[offset:offset + 2] = symbol_value.to_bytes(2, byte_contract['byteorder'])

boot_header = bytearray(b'FT\x00\x00')
patch_abs16(boot_header, 2, 0x0040)
assert boot_header == b'FT@\x00'
artifact = {
    'schema_version': 1,
    'isa': 'toy-fixed-2-byte-v1',
    'byteorder': byte_contract['byteorder'],
    'image_hex': image.hex(),
    'labels': labels,
    'boot_header_hex': boot_header.hex(),
    'entry': 0x0040,
    'uart_output_hex': bytes(result['output']).hex(),
    'trace': result['trace'],
}
target = ARTIFACT_ROOT / 'fromthetransistor/03-machine-image.json'
target.parent.mkdir(parents=True, exist_ok=True)
target.write_text(json.dumps(artifact, indent=2, sort_keys=True) + '\n', encoding='utf-8')
print('machine image bytes:', len(image), 'output:', result['output'])


## Exercise and cross-layer checks

1. Change the first immediate to 7 and predict the UART output before execution.
2. Add an unresolved branch target and require an assembler diagnostic.
3. Relocate an entry above `0xffff` and require a width failure.
4. Extend the monitor trace with memory reads, keeping fetch failure distinct from an illegal opcode.

In [ ]:
image7, _ = assemble(program.replace('MOVI r0, 5', 'MOVI r0, 7'))
assert run(image7)['output'] == [28]
try:
    assemble('JNZ r0, nowhere')
    raise AssertionError('unresolved label accepted')
except ValueError as error:
    assert 'unresolved label' in str(error)
try:
    patch_abs16(bytearray(2), 0, 0x10000)
    raise AssertionError('wide relocation accepted')
except ValueError:
    pass
assert bytes(result['output']).hex() == artifact['uart_output_hex']
print('toolchain self-checks passed')
